## 📋 Lab 7 TPU — To-Do Summary (3 items)

| # | Location | What to implement |
|---|----------|-------------------|
| 8  | CPU benchmark | Average execution time from the collected list |
| 9  | TPU benchmark | Force result materialisation so timing is accurate |
| 10 | Comparison plot | Overlay CPU and GPU results on the same axes |


# CPU version

Run the following: Click on ``Runtime`` menu > Click on ``Change runtime type`` > Choose ``CPU`` under ``Hardware accelerator`` > Click on ``Save``

In [ ]:
import time
import torch

def benchmark_cpu(func, A, B, label, runs=3):
    """Benchmark a matmul function on CPU."""
    times = []
    for _ in range(runs):
        start = time.time()
        func(A, B)
        times.append(time.time() - start)
    # To Do 8: compute the average run time from the list
    # Think about: you have a list of individual run times — what formula gives the mean?
    avg_time = None
    print(f'{label}: {avg_time:.4f} seconds')
    return avg_time


In [ ]:
import torch

cpu = torch.device("cpu")

In [ ]:
MAT = lambda N : torch.randn(N, N)

results = []
for N in range(1024, 8192+1024, 1024):
  results.append(benchmark(torch.matmul, MAT(N), MAT(N), f"N = {N}"))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(1024, 8192+1024, 1024), results, 'o-')
plt.xlabel("Matrix size")
plt.ylabel("Execution time (seconds)")
plt.title("Matrix multiplication execution time")
plt.grid(True)
plt.show()

# TPU version

Run the following: Click on ``Runtime`` menu > Click on ``Change runtime type`` > Choose ``v2-8 TPU`` under ``Hardware accelerator`` > Click on ``Save``

In [ ]:
import time
def benchmark(func, A, B, label, runs=3):
    times = []
    size = A.shape[0] - 1
    for _ in range(runs):
        torch.cuda.empty_cache()
        start = time.time()
        _ = func(A, B)
        print(f"{_[size][size]*_[int(size/2)][int(size/2)]*_[size][int(size/2)]*_[int(size/2)][size]*0}")
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        times.append(time.time() - start)
    avg_time = sum(times) / len(times)
    print(f"{label}: {avg_time:.4f} seconds")
    return avg_time

In [ ]:
import time

def benchmark_tpu(func, A, B, label, runs=3):
    """Benchmark on TPU. Lazy execution means we must force materialisation."""
    times = []
    for _ in range(runs):
        start = time.time()
        result = func(A, B)
        # To Do 9: force the TPU to finish computing before stopping the timer.
        # Think about: TPU tensors are lazy — the computation hasn't actually run yet
        #              after func() returns. You need to read one value to force it.
        #              What method converts a single tensor element to a Python scalar?
        _ = None   # force materialisation here
        times.append(time.time() - start)
    avg_time = sum(times) / len(times)
    print(f'{label}: {avg_time:.4f} seconds')
    return avg_time


In [ ]:
import torch

MAT = lambda N : torch.randn(N, N, device=tpu)

results = []
for N in range(1024, 8192+1024, 1024):
  results.append(benchmark(torch.matmul, MAT(N), MAT(N), f"N = {N}"))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(1024, 8192+1024, 1024), results, 'o-')
plt.xlabel("Matrix size")
plt.ylabel("Execution time (seconds)")
plt.title("Matrix multiplication execution time")
plt.grid(True)
plt.show()

# GPU version

Run the following: Click on ``Runtime`` menu > Click on ``Change runtime type`` > Choose ``GPU T4`` under ``Hardware accelerator`` > Click on ``Save``

In [ ]:
import time
def benchmark(func, A, B, label, runs=3):
    times = []
    for _ in range(runs):
        torch.cuda.empty_cache()
        start = time.time()
        _ = func(A, B)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        times.append(time.time() - start)
    avg_time = sum(times) / len(times)
    print(f"{label}: {avg_time:.4f} seconds")
    return avg_time

In [ ]:
import torch

gpu = torch.device("cuda")

In [ ]:
MAT = lambda N : torch.randn(N, N, device=gpu)

results = []
for N in range(1024, (8192+1024), 1024):
  results.append(benchmark(torch.matmul, MAT(N), MAT(N), f"N = {N}"))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(1024, 8192+1024, 1024), results, 'o-')
plt.xlabel("Matrix size")
plt.ylabel("Execution time (seconds)")
plt.title("Matrix multiplication execution time")
plt.grid(True)
plt.show()

## To Do 10 — Compare CPU, GPU, and TPU on the Same Plot

After running all three sections, use the cell below to overlay the results.
This gives a direct visual comparison of all three hardware targets.


In [ ]:
import matplotlib.pyplot as plt

sizes = list(range(1024, 8192 + 1024, 1024))

# To Do 10: populate these lists with the results from each section above.
# Think about: which variable holds CPU results? GPU results? TPU results?
#              Each section stores results in a list called `results` —
#              but they get overwritten. Re-run each section and save them here.
cpu_results = None   # paste your CPU results list here
gpu_results = None   # paste your GPU results list here
tpu_results = None   # paste your TPU results list here (skip if unavailable)

plt.figure(figsize=(9, 5))
if cpu_results: plt.plot(sizes, cpu_results, 'o-',  label='CPU')
if gpu_results: plt.plot(sizes, gpu_results, 's--', label='GPU (T4)')
if tpu_results: plt.plot(sizes, tpu_results, '^:',  label='TPU v2-8')
plt.xlabel('Matrix size N')
plt.ylabel('Execution time (seconds)')
plt.title('torch.matmul: CPU vs GPU vs TPU')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
